# Phase-1: Data Ingestion & Filtering (England)

**Goal:** Load all raw street crime and outcomes data for all English police forces and save clean outputs as parquet.

**Scope:** All England forces, April 2023 – March 2026

**Outputs saved to:** `England/outputs/phase1/`
- `phase1_crimes_england.parquet`: all street crimes for all English forces
- `phase1_outcomes_england.parquet`: all outcomes for all English forces

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Dataset Loading
BASE  = Path('Dataset path')
OUT   = BASE / 'england_final_light' / 'outputs' / 'phase1'
OUT.mkdir(parents=True, exist_ok=True)

# Crime archive folders in chronological order
ARCHIVES = ['0423-0424', '0524-0525', '0625-0326']

print('Output folder:', OUT)

## Section-1: Crime Data
Load street crimes and outcomes across all 36 months for all English forces. No force filter applied.

In [ ]:
# --- PHASE 1A: Street crimes ---
print('Loading street crime files...')
street_chunks = []

for archive in ARCHIVES:
    archive_path = BASE / 'data' / 'crimes' / archive
    files = sorted(archive_path.rglob('*-street.csv'))
    print(f'  {archive}: {len(files)} files')
    for f in files:
        try:
            df = pd.read_csv(f, dtype=str, low_memory=False)
            if len(df) > 0:
                street_chunks.append(df)
        except Exception as e:
            print(f'  [SKIP] {f.name}: {e}')

crimes = pd.concat(street_chunks, ignore_index=True)
crimes.columns = crimes.columns.str.strip()
crimes['Month'] = pd.to_datetime(crimes['Month'], format='%Y-%m')

print(f'\nShape: {crimes.shape}')
print(f'Date range: {crimes["Month"].min().strftime("%b %Y")} → {crimes["Month"].max().strftime("%b %Y")}')
print(f'Unique forces: {crimes["Falls within"].nunique()}')
print(f'Nulls:\n{crimes.isnull().sum()}')
crimes.head(3)

In [ ]:
# --- PHASE 1B: Outcomes ---
print('Loading outcomes files...')
outcome_chunks = []

for archive in ARCHIVES:
    archive_path = BASE / 'data' / 'crimes' / archive
    files = sorted(archive_path.rglob('*-outcomes.csv'))
    print(f'  {archive}: {len(files)} files')
    for f in files:
        try:
            df = pd.read_csv(f, dtype=str, low_memory=False)
            if len(df) > 0:
                outcome_chunks.append(df)
        except Exception as e:
            print(f'  [SKIP] {f.name}: {e}')

outcomes = pd.concat(outcome_chunks, ignore_index=True)
outcomes.columns = outcomes.columns.str.strip()
outcomes['Month'] = pd.to_datetime(outcomes['Month'], format='%Y-%m')

print(f'\nShape: {outcomes.shape}')
print(f'Date range: {outcomes["Month"].min().strftime("%b %Y")} → {outcomes["Month"].max().strftime("%b %Y")}')
print(f'Nulls:\n{outcomes.isnull().sum()}')
outcomes.head(3)

## Section-2: Save Outputs

In [ ]:
saves = {
    'phase1_crimes_england.parquet':   crimes,
    'phase1_outcomes_england.parquet': outcomes,
}

for filename, df in saves.items():
    path = OUT / filename
    df.to_parquet(path, index=False)
    size_mb = path.stat().st_size / 1e6
    print(f'  Saved {filename} — {df.shape[0]:,} rows x {df.shape[1]} cols ({size_mb:.1f} MB)')

print('\nPhase 1 complete.')

In [ ]:
# Phase-1 Summary
print('=' * 55)
print('PHASE 1 SUMMARY (England)')
print('=' * 55)
print(f'Street crimes (all England):  {len(crimes):>10,} rows')
print(f'Outcomes (all England):       {len(outcomes):>10,} rows')
print(f'Unique forces (crimes):       {crimes["Falls within"].nunique():>10}')
print(f'Date range:                   Apr 2023 → Mar 2026')
print('=' * 55)
print(f'Outputs saved to: {OUT}')